# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# Initialize OpenAI and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [ ]:
# Initialize oolama and constants
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [ ]:
ed = Website("https://edwarddonner.com")
ed.links
#print(ed.get_contents())

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"},
        {"type": "newsletters", "url": "https://another.full.url/newsletters"},
        {"type": "newsletters", "blog": "https://another.full.url/blog"}
    ]
}
"""

In [ ]:
print(link_system_prompt)

In [5]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [ ]:
print(get_links_user_prompt(ed))

In [6]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [ ]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

In [ ]:
get_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [7]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [ ]:
print(get_all_details("https://huggingface.co"))

In [23]:
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."

# Misi - haiku
# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short, Japaneese haiku style brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information. If you get the company logo, print it in as an ascii art"

# Misi - Jesse
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. \
Write the brochure in a style as it would be written by Jesse Pinkman from Breaking Bad. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

In [24]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    return response.choices[0].message.content

In [26]:
brochure_result = create_brochure("HuggingFace", "https://huggingface.co")
display(Markdown(brochure_result))


Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussions', 'url': 'https://discuss.huggingface.co'}, {'type': 'github page', 'url': 'https://github.com/huggingface'}, {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


```markdown
# Yo, Welcome to Hugging Face!

Hey, what's up? So you're curious about Hugging Face, huh? Let me break it down for you in plain Jesse Pinkman style. We’re that dope AI community that's out here building the future, and we're totally all about collaboration—like, if you’re into machine learning and AI, this is the place to be.

## What We Do: AI Magic!

At Hugging Face, we let the machine learning community come together to drop some sick models, datasets, and applications. Seriously, we've got over **1 MILLION models** ready for you to explore. Check out the hottest tools trending this week, like:
- Qwen/Qwen-Image-Edit
- Google’s Gemma
- OpenAI’s GPT-OSS-20B

If you got an idea, it’s time to make it real peeps! We’ve got **400k+ apps** and **250k+ datasets** just waiting for you to dive into.

## Join the Party: Community Vibes

We don't just build—we vibe, baby! Hugging Face is all about giving back to the community. We're like a family, sharing knowledge and supporting each other in this crazy world of AI. Got a portfolio? Build it here with our tools and show off your genius to the world!

## Work With Us: Careers

Looking for a gig? We’re always on the hunt for rad talent to join our crew. If you’re down to innovate with enterprise-grade security and exclusive support, we might just be the spot for you. Plus, we’re chill about remote work—no tie required. Bring your skills, and let’s get to work building the future of AI together.

## Who's Riding with Us?

More than **50,000 organizations** are part of the Hugging Face movement, including:
- AI at Meta
- Google
- Microsoft
- Amazon

These giant companies know what's up. If they trust us, you should too, am I right?

## Why Choose Us?

- **Open Source:** We’re building the foundation of ML tools with the whole community.
- **Unlimited Collaboration:** Host and jam on public models, datasets, and more.
- **All Modalities:** Text, images, videos—you name it, we got it. 
- **Enterprise Solutions:** For those looking to scale big time, we've got you covered.

## Let’s Make It Happen

So, think you're ready to take the plunge? Whether you're here for the community, the models, or the sick opportunities, Hugging Face is where the action's at. 

## Get on Board!

Join the movement! Sign up now and start building your future in AI with us. 

**Hugging Face – We’re more than just AI. We’re a community.**

Peace out.

— Jesse P. (well, sorta)
```


In [17]:
## Translator

In [27]:
def translate_brochure(text, language):
    translator_system_prompt = f"""you are a professional translator, 
    you can translate effectively from English to {language}"""

    translator_user_prompt = f""" Translate the following text to {language} with keeping the original 
    formatting: \n {text}"""
    
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": translator_system_prompt},
            {"role": "user", "content": translator_user_prompt}
          ],
    )
    return response.choices[0].message.content

In [28]:
translate_result = translate_brochure(brochure_result, 'Hungarian')
display(Markdown(translate_result))

```markdown
# Hé, Üdvözlöd a Hugging Face-nél!

Hé, mi újság? Szóval kíváncsi vagy a Hugging Face-re, mi? Hadd bontsam le neked egyszerű Jesse Pinkman stílusban. Mi vagyunk az a menő AI közösség, amely itt építi a jövőt, és teljesen az együttműködésről szólunk – ha érdekel a gépi tanulás és az AI, akkor ez a hely neked való.

## Mit csinálunk: AI varázslat!

A Hugging Face-nél lehetőséget adunk a gépi tanulás közösségének, hogy összegyűljenek, és leverték néhány elképesztő modellt, adatbázist és alkalmazást. Komolyan, több mint **1 MILLIÓ modell** vár rád, hogy felfedezd. Nézd meg a legmenőbb eszközöket, amik ezen a héten népszerűek, mint például:
- Qwen/Qwen-Image-Edit
- Google Gemma
- OpenAI GPT-OSS-20B

Ha van egy ötleted, itt az idő, hogy valóra váltsd, emberek! Több mint **400k+ alkalmazás** és **250k+ adatbázis** vár rád, hogy belevágj.

## Csatlakozz a buliba: Közösségi hangulat

Mi nemcsak építünk – hanem együtt vibrálunk, baby! A Hugging Face teljes mértékben a közösségnek ad vissza. Olyanok vagyunk, mint egy család, aki tudást oszt meg és támogatja egymást ebben a őrült AI világban. Van portfóliód? Építsd itt a mi eszközeinkkel, és mutasd meg a világnak a zsenialitásodat!

## Dolgozz velünk: Karrier

Keresel egy melót? Mindig kutatunk menő tehetségeket, hogy csatlakozzanak a csapatunkhoz. Ha nyitott vagy arra, hogy vállalati szintű biztonsággal és exkluzív támogatással innoválj, lehet, hogy mi vagyunk a megfelelő hely számodra. Plusz, lazán állunk a távmunkához – nyakkendő nem szükséges. Hozd a készségeidet, és kezdjünk el együtt építkezni az AI jövőjén.

## Ki utazik velünk?

Több mint **50,000 szervezet** tagja a Hugging Face mozgalomnak, köztük:
- AI a Metánál
- Google
- Microsoft
- Amazon

Ezek a hatalmas cégek tudják, miről van szó. Ha ők megbíznak bennünket, neked is érdemes, nem igaz?

## Miért válaszd a Hugging Face-t?

- **Nyílt forráskód:** Az ML eszközök alapjait a teljes közösséggel állítjuk össze.
- **Korlátlan együttműködés:** Tarts és jam-el a nyilvános modelleken, adatbázisokon és még sok máson.
- **Minden modality:** Szöveg, képek, videók – amit csak akarsz, nálunk megtalálod.
- **Vállalati megoldások:** Aki nagyban szeretne skálázni, azt meg tudjuk támogatni.

## Csináljuk meg!

Szóval, azt hiszed, készen állsz, hogy belevágj? Akár a közösség, a modellek vagy a menő lehetőségek miatt vagy itt, a Hugging Face a hely, ahol a dolog zajlik.

## Csatlakozz!

Csatlakozz a mozgalomhoz! Iratkozz fel most, és kezdj el építeni a jövődet az AI területén velünk.

**Hugging Face – Több vagyunk, mint AI. Egy közösség vagyunk.**

Tartsd a békét.

— Jesse P. (hát, félig)
```

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>